<a href="https://www.kaggle.com/code/ahmedfakhar123/ml-14-web-scraping-for-machine-learning?scriptVersionId=339679106" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

---
# 🤖 ML Lab 05 — Web Scraping for Machine Learning

### ***Learn how to collect real-world datasets from websites using Python and BeautifulSoup***

---

# 📖 Introduction

In Machine Learning, not all data is available through **CSV files** or **APIs**. Many valuable datasets exist only on websites, making **Web Scraping** an essential skill for collecting real-world data.

Web Scraping is the process of automatically extracting information from web pages using Python. It allows you to gather data such as product details, news articles, job listings, sports statistics, and much more, which can then be used for data analysis and Machine Learning.

In this notebook, you'll learn how to scrape web pages using **Requests** and **BeautifulSoup**, extract the information you need, organize it into a **Pandas DataFrame**, and save it as a CSV file for future use.

> **Note:** Always respect a website's Terms of Service and `robots.txt` rules, and only scrape websites where it is permitted.

<div align="center">
    <img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcR8Xl2D-MF3Z7BotdpzZsadpqM4k_7JG56rSG-l3OkXEw&s=10" width=500>
    <img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRh3I2yI7RuCJcS3CwK_cT3L6QAP7ctCmxd-C47ith1iQ&s=10" width=500>
</div>

---
# 📦 Importing Required Libraries

Before scraping any website, let's import the libraries we'll need.

In [1]:
import pandas as pd
import numpy as np

import requests
from bs4 import BeautifulSoup

---
# 🌐 Sending Our First Request

We'll send an HTTP GET request to the PakWheels website.

To reduce the chances of getting blocked, we'll include a User-Agent header that mimics a real browser.

In [2]:
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/138.0.0.0 Safari/537.36"
    )
}
webpage = requests.get('https://www.pakwheels.com/used-cars/search/-/?q=bmw', headers=headers).text

> **💡 Note:**  
> Some websites block requests made by Python scripts and return a **403 Forbidden** error. If this happens, include a **User-Agent** header in your request to make it appear as though the request is coming from a web browser.

```python
import requests

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/138.0.0.0 Safari/537.36"
    )
}

response = requests.get("YOUR_URL", headers=headers)
```

---
# 🍲 Parsing the HTML

Now let's convert the downloaded HTML into a BeautifulSoup object.

In [3]:
soup = BeautifulSoup(webpage, "lxml")

---
# 👀 Exploring the HTML

Printing the HTML helps us understand the page structure before scraping.

In [4]:
print(soup.prettify())

<html>
 <body>
  <div class="modal" id="safety_precautions">
   <div class="modal-dialog">
    <div class="modal-content">
     <div class="modal-header noborder pb0">
      <button aria-hidden="true" class="close" data-dismiss="modal" type="button">
       ×
      </button>
     </div>
     <div class="modal-body nomargin" style="padding: 0px 40px 10px;">
      <div class="tlc mb25">
       <img height="70px" src="https://wsa1.pakwheels.com/assets/tips-for-safe-deal-3813a40c4961570f7cf04bd9f4c8daa4fc49e2e86c577bcd4236aeef301db2b7.svg"/>
       <div class="mb0 fs18 generic-basic mt10 lhm fwm">
        Tips for Safe Deal
       </div>
      </div>
      <div class="d-flex align-center mb15">
       <img class="mr20" height="36px" src="https://wsa3.pakwheels.com/assets/tip-for-safe-deal-1-dbcc070d37c48c08ad825e2eeccce214cba97e9b1e60b7d6392db3a252181967.svg"/>
       <p class="fs16 nomargin">
        Never make payments in advance.
       </p>
      </div>
      <div class="d-flex align-c

---
# 🔍 Extracting Car Titles

Let's extract all BMW car names.

In [5]:
for title in soup.find_all("h3", style="white-space: normal;"):
    print(title.text.strip())

BMW 2 Series  2021 218i Gran Coupe for Sale
BMW i4  2022 eDrive40 for Sale
BMW X1  2017 sDrive18i A/T for Sale
BMW 5 Series 5th (E60) Generation 2006 525i for Sale
BMW 3 Series  2018 318i for Sale
BMW X1  2018 sDrive18i A/T for Sale
BMW X3  2008  for Sale
Toyota Yaris Hatchback  2021 X for Sale
BMW 2 Series  2020 218i Gran Coupe for Sale
BMW 7 Series  2014 750i for Sale
BMW X1  2017 sDrive18i A/T for Sale
BMW i4  2026 eDrive35 for Sale
BMW 3 Series  1996  for Sale
BMW 5 Series 5th (E60) Generation 2004 525i for Sale
Toyota Hilux  2014 Invincible for Sale
BMW 7 Series 5th (F01) Generation 2009 740i for Sale
BMW X1  2017 sDrive18i A/T for Sale
BMW X1  2017 sDrive18i A/T for Sale
BMW 3 Series 4th (E46) Generation 2000 316i for Sale
BMW X5  2014  for Sale
BMW 7 Series 4th (E65) Generation 2004 745Li for Sale
BMW 3 Series  2013 316i for Sale
BMW X1  2017 sDrive18i A/T for Sale
BMW i5  2024 eDrive40 M Sport for Sale
BMW i5  2023 eDrive40 M Sport for Sale


---
# 🚘 Finding All Car Listings

Each listing is stored inside a `<div>` element with the class well.

In [6]:
all_cars = soup.find_all("div", class_="well")

In [7]:
len(all_cars)

29

---
# 📝 Creating Empty Lists

We'll store every attribute inside a separate list before creating our DataFrame.

In [8]:
name = []
price = []
rating = []
year = []
mileage = []
fuel_type = []
engine = []
transmission = []

---

# 🔄 Scraping One Page

Now we'll extract information from every listing on the current page.

In [9]:
for car in all_cars:
    if car.find("h3") is None:
        continue
    try:
        # Car Name
        name.append(car.find("h3").get_text(strip=True))
    except:
        name.append(np.nan)

    try:
        # Price
        price.append(car.find("div", class_="price-details").get_text(strip=True))
    except:
        price.append(np.nan)

    try:
        # Auction Rating
        rating.append(car.find("span", class_="auction-rating").get_text(strip=True))
    except:
        rating.append(np.nan)

    try:
        specs = car.find("ul", class_="list-unstyled search-vehicle-info-2 fs13").find_all("li")

        if len(specs) >= 5:
            year.append(specs[0].get_text(strip=True))
            mileage.append(specs[1].get_text(strip=True))
            fuel_type.append(specs[2].get_text(strip=True))
            engine.append(specs[3].get_text(strip=True))
            transmission.append(specs[4].get_text(strip=True))

    except:
        year.append(np.nan)
        mileage.append(np.nan)
        fuel_type.append(np.nan)
        engine.append(np.nan)
        transmission.append(np.nan)

---
# 📊 Creating the Dataset

Let's convert the extracted lists into a Pandas DataFrame

In [10]:
cars_df = pd.DataFrame({
    "Car Name": name,
    "Price": price,
    "Auction Rating": rating,
    "Model Year": year,
    "Mileage": mileage,
    "Fuel Type": fuel_type,
    "Engine Capacity": engine,
    "Transmission": transmission,
})
cars_df

,Car Name,Price,Auction Rating,Model Year,Mileage,Fuel Type,Engine Capacity,Transmission
0,BMW 2 Series 2021 218i Gran Coupe for Sale,PKR 1.35crore,9.3/10,2021,"68,064 km",Petrol,1500 cc,Manual
1,BMW i4 2022 eDrive40 for Sale,PKR 2.09crore,NaN,2022,"41,500 km",Electric,84.0 kWh,Automatic
2,BMW X1 2017 sDrive18i A/T for Sale,PKR 72lacs,8.7/10,2017,"89,293 km",Petrol,1500 cc,Automatic
3,BMW 5 Series 5th (E60) Generation 2006 525i fo...,PKR 40lacs,8/10,2006,"89,383 km",Petrol,2500 cc,Automatic
4,BMW 3 Series 2018 318i for Sale,PKR 90lacs,8.3/10,2018,"109,245 km",Petrol,1600 cc,Automatic
5,BMW X1 2018 sDrive18i A/T for Sale,PKR 95lacs,NaN,2018,"150,000 km",Petrol,1500 cc,Automatic
6,BMW X3 2008 for Sale,PKR 40lacs,7.3/10,2008,"93,048 km",Petrol,2500 cc,Automatic
7,Toyota Yaris Hatchback 2021 X for Sale,PKR 47.9lacs,NaN,2021,"80,000 km",Petrol,1000 cc,Automatic
8,BMW 2 Series 2020 218i Gran Coupe for Sale,PKR 1.5crore,9.3/10,2020,"25,441 km",Petrol,1500 cc,Manual
9,BMW 7 Series 2014 750i for Sale,Call,NaN,2014,"48,000 km",Petrol,4400 cc,Automatic


---
# 🚀 Scraping Multiple Pages

Instead of collecting data from only one page, we can loop through all available pages.

In [11]:
all_cars_df = pd.DataFrame()

for page in range(1, 12):
    headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/138.0.0.0 Safari/537.36"
    )}
    webpage = requests.get(f'https://www.pakwheels.com/used-cars/search/-/mk_bmw/?page={page}', headers=headers).text

    soup = BeautifulSoup(webpage, "lxml")

    all_cars = soup.find_all("div", class_="well")

    name = []
    price = []
    rating = []
    year = []
    mileage = []
    fuel_type = []
    engine = []
    transmission = []

    for car in all_cars:
        if car.find("h3") is None:
            continue
        try:
            # Car Name
            name.append(car.find("h3").get_text(strip=True))
        except:
            name.append(np.nan)

        try:
            # Price
            price.append(car.find("div", class_="price-details").get_text(strip=True))
        except:
            price.append(np.nan)

        try:
            # Auction Rating
            rating.append(car.find("span", class_="auction-rating").get_text(strip=True))
        except:
            rating.append(np.nan)

        try:
            specs = car.find("ul", class_="list-unstyled search-vehicle-info-2 fs13").find_all("li")

            if len(specs) >= 5:
                year.append(specs[0].get_text(strip=True))
                mileage.append(specs[1].get_text(strip=True))
                fuel_type.append(specs[2].get_text(strip=True))
                engine.append(specs[3].get_text(strip=True))
                transmission.append(specs[4].get_text(strip=True))

        except:
            year.append(np.nan)
            mileage.append(np.nan)
            fuel_type.append(np.nan)
            engine.append(np.nan)
            transmission.append(np.nan)

        current_cars_df = pd.DataFrame({
        "Car Name": name,
        "Price": price,
        "Auction Rating": rating,
        "Model Year": year,
        "Mileage": mileage,
        "Fuel Type": fuel_type,
        "Engine Capacity": engine,
        "Transmission": transmission,
        })

    all_cars_df = pd.concat([all_cars_df, current_cars_df], ignore_index=True)

In [12]:
all_cars_df

,Car Name,Price,Auction Rating,Model Year,Mileage,Fuel Type,Engine Capacity,Transmission
0,BMW 2 Series 2021 218i Gran Coupe for Sale,PKR 1.35crore,9.3/10,2021,"68,064 km",Petrol,1500 cc,Manual
1,BMW i4 2022 eDrive40 for Sale,PKR 2.09crore,NaN,2022,"41,500 km",Electric,84.0 kWh,Automatic
2,BMW X1 2017 sDrive18i A/T for Sale,PKR 72lacs,8.7/10,2017,"89,293 km",Petrol,1500 cc,Automatic
3,BMW 5 Series 5th (E60) Generation 2006 525i fo...,PKR 40lacs,8/10,2006,"89,383 km",Petrol,2500 cc,Automatic
4,BMW 3 Series 2018 318i for Sale,PKR 90lacs,8.3/10,2018,"109,245 km",Petrol,1600 cc,Automatic
...,...,...,...,...,...,...,...,...
353,BMW 3 Series 2013 for Sale,PKR 80lacs,NaN,2013,"70,000 km",Petrol,1600 cc,Manual
354,BMW M5 2019 for Sale,PKR 4.55crore,NaN,2019,"59,000 km",Petrol,4400 cc,Automatic
355,BMW X6 2009 for Sale,PKR 1.6crore,NaN,2009,"104,000 km",Diesel,3000 cc,Automatic
356,BMW 3 Series 1980 for Sale,PKR 90lacs,NaN,1980,2 km,Petrol,4400 cc,Automatic


---
# 🔄 Converting Data Types

The scraped data is currently stored as **`object` (string)** because websites return almost all values as text. However, machine learning models and data analysis require numerical values for calculations, filtering, and visualization.

In this step, we'll convert each column to its appropriate data type by cleaning unnecessary text (such as `PKR`, `km`, `cc`, and `/10`) and transforming the values into numeric or categorical formats.

### Changes Made

| Column          | Current | Will be           |
| --------------- | ------- | ------------------- |
| Car Name        | object  | object              |
| Price           | object  | numeric (float/int) |
| Auction Rating  | object  | float               |
| Model Year      | object  | int                 |
| Mileage         | object  | int                 |
| Fuel Type       | object  | category/object     |
| Engine Capacity | object  | numeric             |
| Transmission    | object  | category/object     |


After these conversions, the dataset becomes cleaner, more memory-efficient, and ready for preprocessing, exploratory data analysis (EDA), and machine learning.

## 1. Model Year → int

In [13]:
all_cars_df["Model Year"] = pd.to_numeric(
    all_cars_df["Model Year"],
    errors="coerce"
).astype("Int64")

## 2. Auction Rating → float

In [14]:
all_cars_df["Auction Rating"] = (
    all_cars_df["Auction Rating"]
    .str.replace("/10", "", regex=False)
)

all_cars_df["Auction Rating"] = pd.to_numeric(
    all_cars_df["Auction Rating"],
    errors="coerce"
)

## 3. Mileage → int

In [15]:
all_cars_df["Mileage"] = (
    all_cars_df["Mileage"]
    .str.replace(",", "", regex=False)
    .str.replace(" km", "", regex=False)
)

all_cars_df["Mileage"] = pd.to_numeric(
    all_cars_df["Mileage"],
    errors="coerce"
).astype("Int64")

## 4. Engine Capacity

This one is tricky because we have
```
1600 cc
1500 cc
81.0 kWh
710.0 kWh
```
Since EVs use kWh and fuel cars use cc, we don't combine them into one numeric column.

In [16]:
all_cars_df["Engine Unit"] = (
    all_cars_df["Engine Capacity"]
    .str.extract(r'(cc|kWh)')
)

all_cars_df["Engine Capacity"] = (
    all_cars_df["Engine Capacity"]
    .str.extract(r'([\d.]+)')
)

all_cars_df["Engine Capacity"] = pd.to_numeric(
    all_cars_df["Engine Capacity"],
    errors="coerce"
)

## 5. Price

In [17]:
def convert_price(price):
    if pd.isna(price):
        return None

    price = price.lower().replace("pkr", "").strip()

    if "crore" in price:
        return float(price.replace("crore", "")) * 10_000_000

    elif "lac" in price:
        price = price.replace("lacs", "").replace("lac", "")
        return float(price) * 100_000

    return None

all_cars_df["Price (PKR)"] = all_cars_df["Price"].apply(convert_price)

all_cars_df.drop("Price", inplace=True, axis=1)

## 6. Fuel Type & Transmission

In [18]:
all_cars_df["Fuel Type"] = all_cars_df["Fuel Type"].astype("category")
all_cars_df["Transmission"] = all_cars_df["Transmission"].astype("category")

In [19]:
all_cars_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 358 entries, 0 to 357
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   Car Name         358 non-null    object  
 1   Auction Rating   26 non-null     float64 
 2   Model Year       358 non-null    Int64   
 3   Mileage          358 non-null    Int64   
 4   Fuel Type        358 non-null    category
 5   Engine Capacity  358 non-null    float64 
 6   Transmission     358 non-null    category
 7   Engine Unit      358 non-null    object  
 8   Price (PKR)      307 non-null    float64 
dtypes: Int64(2), category(2), float64(3), object(2)
memory usage: 21.4+ KB


---
# 🎉 Final Scraped Dataset

The complete dataset has now been collected by combining data from all scraped pages into a single Pandas DataFrame.

In [20]:
all_cars_df

,Car Name,Auction Rating,Model Year,Mileage,Fuel Type,Engine Capacity,Transmission,Engine Unit,Price (PKR)
0,BMW 2 Series 2021 218i Gran Coupe for Sale,9.3,2021,68064,Petrol,1500.0,Manual,cc,13500000.0
1,BMW i4 2022 eDrive40 for Sale,NaN,2022,41500,Electric,84.0,Automatic,kWh,20900000.0
2,BMW X1 2017 sDrive18i A/T for Sale,8.7,2017,89293,Petrol,1500.0,Automatic,cc,7200000.0
3,BMW 5 Series 5th (E60) Generation 2006 525i fo...,8.0,2006,89383,Petrol,2500.0,Automatic,cc,4000000.0
4,BMW 3 Series 2018 318i for Sale,8.3,2018,109245,Petrol,1600.0,Automatic,cc,9000000.0
...,...,...,...,...,...,...,...,...,...
353,BMW 3 Series 2013 for Sale,NaN,2013,70000,Petrol,1600.0,Manual,cc,8000000.0
354,BMW M5 2019 for Sale,NaN,2019,59000,Petrol,4400.0,Automatic,cc,45500000.0
355,BMW X6 2009 for Sale,NaN,2009,104000,Diesel,3000.0,Automatic,cc,16000000.0
356,BMW 3 Series 1980 for Sale,NaN,1980,2,Petrol,4400.0,Automatic,cc,9000000.0


---
# 💾 Saving the Dataset

Finally, save the scraped data for future analysis or Machine Learning projects.

In [21]:
all_cars_df.to_csv("bmw_cars_pakistan.csv")

---
# 🎯 Conclusion

Congratulations! 🎉 In this notebook, you learned how to collect real-world data by scraping the PakWheels website using **Requests** and **BeautifulSoup**. Starting from sending HTTP requests with browser headers to parsing HTML, extracting useful information, and combining data from multiple pages into a single DataFrame, you've built a complete web scraping workflow.

The final dataset is now ready for the next stages of the data science pipeline, including **data cleaning, exploratory data analysis (EDA), feature engineering, and machine learning**. Web scraping is a valuable skill that enables you to gather custom datasets when publicly available data isn't sufficient for your projects.

Keep in mind that website structures can change over time, so scraping code may need updates. Always respect a website's terms of service and scrape responsibly. 🚀


---
# 🚀 What's Next?

[**🤖 ML 15 — Automated EDA with YData Profiling**](https://www.kaggle.com/code/ahmedfakhar123/ml-15-automated-eda-with-ydata-profiling)

Awesome work! 🎉 You've successfully collected real-world data through web scraping. Now it's time to explore your dataset automatically and uncover insights in just a few lines of code.

In the next notebook, you'll learn how to:

- 🤖 Generate a complete Exploratory Data Analysis (EDA) report automatically
- 📊 Explore dataset statistics, distributions, and correlations
- ❓ Detect missing values and duplicate records
- 📈 Analyze numerical and categorical features
- 🔍 Identify potential data quality issues
- ⚠️ Discover outliers and unusual patterns
- 🧠 Gain valuable insights before data preprocessing and model building

By the end, you'll be able to create professional, interactive EDA reports with **YData Profiling**, making dataset exploration faster, easier, and more insightful.

Happy Coding! 🚀